In [ ]:
!pip install -q peft transformers datasets accelerate ftfy regex Pillow tqdm


In [ ]:
import os
import gc
import math
import random
import json
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPTokenizer
import copy

from peft import LoraConfig, get_peft_model
from datasets import load_dataset, concatenate_datasets
from tqdm.auto import tqdm

# ── Memory-efficiency settings for bigG (ViT-bigG-14) ───────────────────────
# T4 supports TF32 for matmul — roughly 2× throughput with negligible precision loss
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

# ── Detect dual T4 ──────────────────────────────────────────────────────────
N_GPU  = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPUs available: {N_GPU}  |  device: {DEVICE}")


In [ ]:
# ── Random permutation + word rotation ──────────────────────────────────────
ALPHABET = "abcdefghijklmnopqrstuvwxyz"

KEY = "cixmpuqbstreyofwlkzhgnavdj"   # <── Must match Phase 1 exactly

def _validate_key(key: str) -> str:
    key = key.strip().lower()
    assert sorted(key) == list(ALPHABET), (
        "KEY must be a permutation of all 26 lowercase letters. "
    )
    return key

KEY = _validate_key(KEY)

_ENC_TABLE = str.maketrans(ALPHABET + ALPHABET.upper(), KEY + KEY.upper())
_DEC_TABLE = str.maketrans(KEY + KEY.upper(), ALPHABET + ALPHABET.upper())

def _rotate(word: str) -> str:
    return (word[-1] + word[:-1]) if len(word) > 1 else word

def _unrotate(word: str) -> str:
    return (word[1:] + word[0]) if len(word) > 1 else word

def encrypt_prompt(prompt: str) -> str:
    return " ".join(_rotate(w.translate(_ENC_TABLE)) for w in prompt.split())

def decrypt_prompt(cipher: str) -> str:
    return " ".join(_unrotate(w).translate(_DEC_TABLE) for w in cipher.split())

# Sanity check
_EXAMPLES = ["a cat on a mat", "hello world", "a beautiful sunset over the ocean"]
print("── Cipher sanity check ──────────────────────────────────")
for p in _EXAMPLES:
    c = encrypt_prompt(p)
    d = decrypt_prompt(c)
    print(f"  plain : {p}\n  cipher: {c}\n  back  : {d}  {'✓' if d == p else '✗'}\n")


In [ ]:
# ── Single source of truth for the model ────────────────────────────────────
CLIP_MODEL_ID = "laion/CLIP-ViT-bigG-14-laion2B-39B-b160k"

MAX_SAMPLES   = 1_000_000
MAX_TOKEN_LEN = 77
CACHE_DIR     = "/kaggle/working/data_cache"
PAIRS_PATH    = Path("/kaggle/working/pairs_phase2.jsonl")

pairs = []

# ── [SPECIFIC REQUIREMENT] Check if local pre-built pairs exist ──────────────
if PAIRS_PATH.exists():
    print(f"[INFO] Found existing local file: {PAIRS_PATH}")
    print("       Loading pairs directly from disk to bypass download and text generation...")
    with open(PAIRS_PATH, "w" if not PAIRS_PATH.exists() else "r") as f:
        for line in f:
            row = json.loads(line)
            pairs.append((row["plain"], row["cipher"]))
    print(f"[SUCCESS] Loaded {len(pairs):,} pre-computed prompt pairs.")
else:
    print(f"[INFO] {PAIRS_PATH} not found. Proceeding with downloading and generating text pairs...")
    
    # ── Exclude Phase 1 val pairs to prevent cross-phase data leakage ──
    PHASE1_PAIRS_PATH = "/kaggle/input/datasets/kristofprogamer/sdxl-phas1-pairs/pairs.jsonl"
    phase1_val_captions = set()
    if PHASE1_PAIRS_PATH and Path(PHASE1_PAIRS_PATH).exists():
        p1_all = []
        with open(PHASE1_PAIRS_PATH) as f:
            for line in f:
                row = json.loads(line)
                p1_all.append(row["plain"])
        _tmp_rng = random.Random(42)
        _tmp_rng.shuffle(p1_all)
        n_val_p1 = max(1, int(len(p1_all) * 0.02))
        phase1_val_captions = set(p1_all[:n_val_p1])
        print(f"Loaded {len(phase1_val_captions):,} Phase 1 validation captions to exclude.")
    else:
        print("WARNING: PHASE1_PAIRS_PATH not found. Phase 1 pairs will not be excluded.")

    print("Loading SD Prompts corpus from HuggingFace...")
    sd_source = load_dataset(
        "daspartho/stable-diffusion-prompts", split="train",
        cache_dir=CACHE_DIR,
    )

    def get_text(row):
        return str(row.get("Prompt", row.get("prompt", row.get("text", ""))))

    print(f"Dataset loaded. Total rows available: {len(sd_source):,}")

    def build_pairs(dataset, max_samples: int, exclude: set = None):
        _tok = CLIPTokenizer.from_pretrained(CLIP_MODEL_ID)
        pairs_list, seen = [], set()
        exclude = exclude or set()

        for row in tqdm(dataset, total=min(len(dataset), max_samples), desc="Building pairs"):
            cap = get_text(row).strip()
            if not cap or cap in seen or cap in exclude:
                continue
            seen.add(cap)

            cipher = encrypt_prompt(cap)

            if (len(_tok(cap,    truncation=False)["input_ids"]) <= MAX_TOKEN_LEN and
                len(_tok(cipher, truncation=False)["input_ids"]) <= MAX_TOKEN_LEN):
                pairs_list.append((cap, cipher))

            if len(pairs_list) >= max_samples:
                break
        return pairs_list

    pairs = build_pairs(sd_source, max_samples=MAX_SAMPLES, exclude=phase1_val_captions)
    print(f"\nGenerated {len(pairs):,} (plain, cipher) pairs for extended training.")

    with open(PAIRS_PATH, "w") as f:
        for p, c in pairs:
            f.write(json.dumps({"plain": p, "cipher": c}) + "\n")
    print(f"Saved generated pairs to local file: {PAIRS_PATH}")

In [ ]:
# ── [FIXED OOM] Batch Size & Accumulation Scaling ───────────────────────────
# The text encoder of ViT-bigG-14 contains ~740M parameters. Running a physical
# batch size of 128 (64/GPU) causes an instant OutOfMemoryError during training 
# because of backpropagation activation maps. 
# We lower the physical batch size to 32 (16 per GPU) and scale gradient 
# accumulation steps to 8. This guarantees an identical effective batch size 
# (32 × 8 = 256) and identical contrastive physics without exceeding the VRAM limit.
# ─────────────────────────────────────────────────────────────────────────────
BATCH_SIZE  = 32    # Physical micro-batch size (16 sequences/GPU on dual T4)
ACCUM_STEPS = 8     # Accumulation steps to preserve an effective batch of 256
VAL_FRAC    = 0.02
SEED        = 42

# ── Ensure Reproducibility ───────────────────────────────────────────────────
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

random.shuffle(pairs)

n_val       = max(1, int(len(pairs) * VAL_FRAC))
val_pairs   = pairs[:n_val]
train_pairs = pairs[n_val:]
print(f"Train samples: {len(train_pairs):,}  |  Val samples: {len(val_pairs):,}")

class CipherDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=77):
        self.pairs     = pairs
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        plain, cipher = self.pairs[idx]
        enc_p = self.tokenizer(plain,  padding="max_length", truncation=True,
                               max_length=self.max_len, return_tensors="pt")
        enc_c = self.tokenizer(cipher, padding="max_length", truncation=True,
                               max_length=self.max_len, return_tensors="pt")
        return {
            "plain_input_ids":      enc_p["input_ids"].squeeze(0),
            "plain_attention_mask": enc_p["attention_mask"].squeeze(0),
            "cipher_input_ids":     enc_c["input_ids"].squeeze(0),
            "cipher_attention_mask":enc_c["attention_mask"].squeeze(0),
        }

tokenizer    = CLIPTokenizer.from_pretrained(CLIP_MODEL_ID)
train_ds     = CipherDataset(train_pairs, tokenizer)
val_ds       = CipherDataset(val_pairs,   tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

In [ ]:
from peft import PeftModel

# ── [RESUME CONFIGURATION] ──────────────────────────────────────────────────
# Set these to the exact point where the session died.
# Example: If your last saved checkpoint was 'lora_bigG_epoch6_step500'
START_EPOCH = 6
RESUME_STEP = 500  
# ─────────────────────────────────────────────────────────────────────────────

# ── Precision note ──────────────────────────────────────────────────────────
clip_full = CLIPModel.from_pretrained(
    CLIP_MODEL_ID,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)
clip_full.eval()

# ── Reference encoder ───────────────────────────────────────────────────────
ref_encoder = copy.deepcopy(clip_full.text_model).float()
ref_proj    = copy.deepcopy(clip_full.text_projection).float()
for p in ref_encoder.parameters(): p.requires_grad_(False)
for p in ref_proj.parameters():    p.requires_grad_(False)
ref_encoder.eval()
ref_encoder = ref_encoder.to(DEVICE)
ref_proj    = ref_proj.to(DEVICE)

# ── Resume bigG LoRA encoder from the interrupted checkpoint ────────────────
if RESUME_STEP > 0:
    # IMPORTANT: If you are in a NEW session, you must add the previous run's 
    # output as a dataset. Change '/kaggle/working/' to '/kaggle/input/[dataset-name]/'
    CKPT_PATH = f"/kaggle/working/checkpoints_bigG_phase2/lora_bigG_epoch{START_EPOCH}_step{RESUME_STEP}"
else:
    # Starting fresh from Phase 1 epoch 5
    CKPT_PATH = "/kaggle/input/models/kristofprogamer/best-lora-epoch5-cos0-9095/transformers/default/2"

print(f"Loading LoRA weights from: {CKPT_PATH}")

lora_encoder = PeftModel.from_pretrained(
    clip_full.text_model.float(),
    CKPT_PATH,
    is_trainable=True,
)

lora_proj = copy.deepcopy(clip_full.text_projection).float()
for p in lora_proj.parameters(): p.requires_grad_(False)

lora_encoder.print_trainable_parameters()
n_trainable = sum(p.numel() for p in lora_encoder.parameters() if p.requires_grad)
assert n_trainable > 0, "FATAL: lora_encoder has no trainable parameters!"

if N_GPU > 1:
    lora_encoder = nn.DataParallel(lora_encoder)
    ref_encoder  = nn.DataParallel(ref_encoder)

lora_encoder = lora_encoder.to(DEVICE)
lora_proj    = lora_proj.to(DEVICE)

del clip_full
gc.collect()
torch.cuda.empty_cache()
print("Full bigG model freed from memory.")

In [ ]:
def cosine_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def infonce_loss(pred: torch.Tensor, target: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    p = F.normalize(pred,   dim=-1)
    t = F.normalize(target, dim=-1)
    logits  = (p @ t.T) / temperature
    targets = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, targets)

def combined_loss(pred, target, alpha=0.7):
    return alpha * cosine_loss(pred, target) + (1 - alpha) * infonce_loss(pred, target)


In [ ]:
def _unwrap(module):
    return module.module if isinstance(module, nn.DataParallel) else module

def embed_plain(input_ids, attention_mask):
    with torch.no_grad():
        out = ref_encoder(input_ids=input_ids, attention_mask=attention_mask)
        emb = _unwrap(ref_proj)(out.pooler_output)
    return F.normalize(emb, dim=-1)

def embed_cipher(input_ids, attention_mask):
    out = lora_encoder(input_ids=input_ids, attention_mask=attention_mask)
    emb = _unwrap(lora_proj)(out.pooler_output)
    return F.normalize(emb, dim=-1)


**bigG rough timing guide (dual T4)**  
ViT-bigG text encoder is ~5× heavier than ViT-L.  
Expect ~8–12 s per optimiser step (128 × 2 = 256 effective batch).  
~3 800 opt-steps/epoch → ~9–12 h/epoch on dual T4.  
Use Kaggle's 9-hour GPU sessions with checkpoint-resume to span multiple sessions.


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
LR         = 3e-5    
EPOCHS     = 3       # Total epochs planned (Epochs 6–8)
WARMUP     = 500     
LOG_EVERY  = 250     
SAVE_EVERY = 500    
CKPT_DIR   = Path("/kaggle/working/checkpoints_bigG_phase2")
CKPT_DIR.mkdir(exist_ok=True)

trainable = list(filter(lambda p: p.requires_grad, lora_encoder.parameters()))
print(f"Total trainable tensors in optimizer: {len(trainable)}")
print(f"Total trainable parameters          : {sum(p.numel() for p in trainable):,}")

optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-4)

steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
total_steps     = EPOCHS * steps_per_epoch

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=total_steps,
    pct_start=WARMUP / total_steps, anneal_strategy="cos",
)
scaler = torch.amp.GradScaler("cuda")

# ── FAST-FORWARD SCHEDULER ───────────────────────────────────────────────────
if RESUME_STEP > 0 or START_EPOCH > 6:
    total_steps_completed = ((START_EPOCH - 6) * steps_per_epoch) + RESUME_STEP 
    print(f"Fast-forwarding scheduler by {total_steps_completed} steps to maintain LR curve...")
    for _ in range(total_steps_completed):
        scheduler.step()
# ─────────────────────────────────────────────────────────────────────────────

best_val_cos = 0.9095
history      = {"train_loss": [], "val_cos": []}

for epoch in range(START_EPOCH, 6 + EPOCHS): 
    lora_encoder.train()
    running_loss = 0.0
    epoch_loss = 0.0
    skipped_steps = 0
    
    # Set opt_step to where we left off if resuming mid-epoch
    opt_step = RESUME_STEP if epoch == START_EPOCH else 0
    
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader, 1):
        # ── SKIP COMPLETED BATCHES (Dataloader Fast-Forward) ─────────────────
        if epoch == START_EPOCH and step <= (RESUME_STEP * ACCUM_STEPS):
            if step % (LOG_EVERY * ACCUM_STEPS) == 0:
                print(f" Fast-forwarding dataloader... skipping batch {step}/{RESUME_STEP * ACCUM_STEPS}")
            continue
        # ─────────────────────────────────────────────────────────────────────

        plain_ids  = batch["plain_input_ids"].to(DEVICE, non_blocking=True)
        plain_mask = batch["plain_attention_mask"].to(DEVICE, non_blocking=True)
        ciph_ids   = batch["cipher_input_ids"].to(DEVICE, non_blocking=True)
        ciph_mask  = batch["cipher_attention_mask"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda"):
            with torch.no_grad():
                target_emb = embed_plain(plain_ids, plain_mask)
            
            pred_emb = embed_cipher(ciph_ids, ciph_mask)
            loss = combined_loss(pred_emb, target_emb) / ACCUM_STEPS

        scaler.scale(loss).backward()

        loss_val = loss.item() * ACCUM_STEPS
        running_loss += loss_val
        epoch_loss += loss_val

        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            opt_step += 1
            scale_before = scaler.get_scale()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scale_after = scaler.get_scale()

            if scale_after >= scale_before:
                scheduler.step()
            else:
                skipped_steps += 1

            optimizer.zero_grad(set_to_none=True)

            # ── Logging ──
            if opt_step % LOG_EVERY == 0:
                avg = running_loss / (LOG_EVERY * ACCUM_STEPS)
                running_loss = 0.0
                print(f" Epoch {epoch} | opt_step {opt_step:>5}/{steps_per_epoch} "
                      f"| loss {avg:.4f} | lr {scheduler.get_last_lr()[0]:.2e}"
                      + (f" | skipped {skipped_steps}" if skipped_steps else ""))
            
            # ── Mid-Epoch Saving ──
            if opt_step % SAVE_EVERY == 0:
                step_ckpt_path = CKPT_DIR / f"lora_bigG_epoch{epoch}_step{opt_step}"
                _unwrap(lora_encoder).save_pretrained(str(step_ckpt_path))
                print(f" ✓ Saved mid-epoch checkpoint → {step_ckpt_path}")

    # ── Validation (End of Epoch) ─────────────────────────────────────────────
    lora_encoder.eval()
    cos_sum, n_batches = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            plain_ids  = batch["plain_input_ids"].to(DEVICE, non_blocking=True)
            plain_mask = batch["plain_attention_mask"].to(DEVICE, non_blocking=True)
            ciph_ids   = batch["cipher_input_ids"].to(DEVICE, non_blocking=True)
            ciph_mask  = batch["cipher_attention_mask"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda"):
                cos_sum += F.cosine_similarity(
                    embed_cipher(ciph_ids, ciph_mask),
                    embed_plain(plain_ids, plain_mask), dim=-1
                ).mean().item()
                n_batches += 1

    val_cos = cos_sum / n_batches
    
    # Calculate active steps to avoid diluting the average loss with skipped batches
    active_opt_steps = max(1, steps_per_epoch - (RESUME_STEP if epoch == START_EPOCH else 0))
    history["train_loss"].append(epoch_loss / active_opt_steps)
    history["val_cos"].append(val_cos)
    print(f"\n── Epoch {epoch} complete ── val cosine similarity: {val_cos:.4f}\n")

    # ── Checkpoint Saving (End of Epoch) ─────────────────────────────────────
    ckpt_path = CKPT_DIR / f"lora_bigG_epoch{epoch}_cos{val_cos:.4f}"
    _unwrap(lora_encoder).save_pretrained(str(ckpt_path))
    print(f" ✓ Saved checkpoint → {ckpt_path}")

    if val_cos > best_val_cos:
        best_val_cos = val_cos
        best_path = CKPT_DIR / f"best_lora_bigG_epoch{epoch}_cos{val_cos:.4f}"
        _unwrap(lora_encoder).save_pretrained(str(best_path))
        print(f" ★ New best! ({val_cos:.4f}) → {best_path}\n")

    gc.collect()
    torch.cuda.empty_cache()

print(f"\nPhase 2 Training complete. Best val cosine similarity: {best_val_cos:.4f}")

In [ ]:
lora_encoder.eval()

test_prompts = [
    "a cat sitting on a red mat",
    "futuristic city at night with neon lights",
    "intricate sensual corset lingerie highly detailed",
]

print("══ Qualitative embedding alignment ══════════════════════")
with torch.no_grad():
    for prompt in test_prompts:
        cipher = encrypt_prompt(prompt)
        p_ids  = tokenizer(prompt, return_tensors="pt", padding="max_length", max_length=77, truncation=True).to(DEVICE)
        c_ids  = tokenizer(cipher, return_tensors="pt", padding="max_length", max_length=77, truncation=True).to(DEVICE)
        cos = F.cosine_similarity(
            embed_cipher(c_ids["input_ids"],  c_ids["attention_mask"]),
            embed_plain( p_ids["input_ids"],  p_ids["attention_mask"]), dim=-1
        ).item()
        print(f"  [{cos:.4f}]  plain : {prompt}\n           cipher: {cipher}\n")

cos_vals = []
with torch.no_grad():
    for batch in val_loader:
        plain_ids  = batch["plain_input_ids"].to(DEVICE, non_blocking=True)
        plain_mask = batch["plain_attention_mask"].to(DEVICE, non_blocking=True)
        ciph_ids   = batch["cipher_input_ids"].to(DEVICE, non_blocking=True)
        ciph_mask  = batch["cipher_attention_mask"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda"):
            cos_vals.extend(
                F.cosine_similarity(
                    embed_cipher(ciph_ids, ciph_mask),
                    embed_plain(plain_ids, plain_mask), dim=-1
                ).tolist()
            )

cos_t = torch.tensor(cos_vals)
print("══ Validation set cosine similarity ══════════════════════")
print(f"  mean  : {cos_t.mean():.4f}")
print(f"  std   : {cos_t.std():.4f}")
print(f"  min   : {cos_t.min():.4f}")
print(f"  > 0.90: {(cos_t > 0.90).float().mean()*100:.1f} %")
print(f"  > 0.95: {(cos_t > 0.95).float().mean()*100:.1f} %")


In [ ]:
# Analytics
def letter_freq(text: str) -> dict:
    text  = text.lower()
    total = sum(1 for c in text if c.isalpha())
    if not total: return {}
    return {c: text.count(c) / total for c in ALPHABET}

cipher_corpus = " ".join(c for _, c in pairs[:10_000])
plain_corpus  = " ".join(p for p, _ in pairs[:10_000])

cf_cipher = letter_freq(cipher_corpus)
cf_plain  = letter_freq(plain_corpus)

uniform   = 1 / 26
kl_cipher = sum(v * math.log(v / uniform) for v in cf_cipher.values() if v > 0)
kl_plain  = sum(v * math.log(v / uniform) for v in cf_plain.values()  if v > 0)

print("══ Frequency analysis (security check) ═══════════════════")
print(f"  KL(plain  ‖ uniform) = {kl_plain:.4f}\n  KL(cipher ‖ uniform) = {kl_cipher:.4f}")
print("\n  Top-5 plain  letters: ", sorted(cf_plain.items(),  key=lambda x: -x[1])[:5])
print("  Top-5 cipher letters: ", sorted(cf_cipher.items(), key=lambda x: -x[1])[:5])

# [FIX #8] Distinct filenames — Phase 1's outputs are not overwritten.
with open("/kaggle/working/training_history_phase2.json", "w") as f:
    json.dump(history, f, indent=2)

with open("/kaggle/working/sample_pairs_phase2.jsonl", "w") as f:
    for p, c in pairs[:1000]:
        f.write(json.dumps({"plain": p, "cipher": c}) + "\n")

print("\nOutputs written to /kaggle/working/ -> checkpoints_bigG_phase2, training_history_phase2.json, sample_pairs_phase2.jsonl")


In [ ]:
import os
import zipfile
from pathlib import Path

WORKING_DIR    = Path("/kaggle/working")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints_bigG_phase2"
OUTPUT_ZIP     = WORKING_DIR / "checkpoints_bigG_phase2.zip"

def zip_folder(folder: Path, output: Path):
    if not folder.exists():
        candidates = [
            d for d in WORKING_DIR.iterdir()
            if d.is_dir() and any(
                kw in d.name.lower()
                for kw in ("checkpoint", "ckpt", "model", "output", "lora", "bigg")
            )
        ]
        if not candidates:
            print(f"[ERROR] No checkpoint folder found in {WORKING_DIR}")
            print("Contents:", [d.name for d in WORKING_DIR.iterdir()])
            return
        folder = candidates[0]
        print(f"[INFO] Auto-detected checkpoint folder: {folder}")

    print(f"[INFO] Zipping: {folder}  →  {output}")
    file_count = 0

    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in folder.rglob("*"):
            if file.is_file():
                arcname = file.relative_to(folder.parent)
                zf.write(file, arcname)
                file_count += 1
                print(f"  + {arcname}")

    size_mb = output.stat().st_size / 1024 / 1024
    print(f"\n[DONE] {file_count} files → {output.name} ({size_mb:.1f} MB)")

zip_folder(CHECKPOINT_DIR, OUTPUT_ZIP)
